# Map Stockholm POIs to stadsdelar (2021 boundaries)

Maps POI latitude/longitude to **stadsdel** (smaller geographic districts) using Stockholm City boundary data archived in [krasch/stockholm-stadsdelar](https://github.com/krasch/stockholm-stadsdelar). The polygons reflect **2021**, not the current administrative arrangement. This boundary GeoJSON contains a `name` attribute for each smaller district, but **does not include a larger administrative-area field**, so none is invented here.

The boundary file excludes water areas. Any points in water or outside the boundaries remain unassigned. Run all cells with internet access the first time to download the boundary file, or manually place the linked GeoJSON beside this notebook.

In [1]:
# Run only if dependencies are missing:
# %pip install geopandas pandas requests shapely pyproj

In [2]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import requests

INPUT = Path("stockholm_poi_enriched_analysis.csv")
OUTPUT = Path("stockholm_poi_with_districts.csv")
BOUNDARIES = Path("stockholm_stadsdelar_2021.geojson")
URL = ("https://raw.githubusercontent.com/krasch/stockholm-stadsdelar/"
       "master/geojson/simplified/stadsdelar_utan_vatten.geo.json")

if not BOUNDARIES.exists():
    response = requests.get(URL, timeout=90)
    response.raise_for_status()
    BOUNDARIES.write_bytes(response.content)

pois = pd.read_csv(INPUT, low_memory=False)
polygons = gpd.read_file(BOUNDARIES).to_crs("EPSG:4326")
print("Boundary fields:", polygons.columns.tolist())
assert "name" in polygons.columns, "Expected district names under the 'name' field."
polygons = polygons[["name", "geometry"]].rename(columns={"name": "district"})
polygons["district"] = polygons["district"].astype("string").str.strip().str.title()
print(f"POIs: {len(pois):,} | district polygons: {len(polygons)}")

Boundary fields: ['name', 'geometry']
POIs: 4,234 | district polygons: 117


In [3]:
# Spatial join. 'intersects' includes points precisely on the boundary.
# Resetting a numeric row ID lets us safely restore the original POI order.
pois = pois.reset_index(drop=True)
pois["_poi_row"] = pois.index
points = gpd.GeoDataFrame(
    pois,
    geometry=gpd.points_from_xy(pois["longitude"], pois["latitude"]),
    crs="EPSG:4326"
)
joined = gpd.sjoin(points, polygons, how="left", predicate="intersects")

# If a point touches multiple polygons, keep one and flag it for inspection.
match_counts = joined.groupby("_poi_row")["district"].count()
ambiguous = (match_counts > 1).sum()
joined = joined.sort_values(["_poi_row", "district"]).drop_duplicates("_poi_row")

result = pd.DataFrame(joined.drop(columns=["geometry", "index_right"], errors="ignore"))
result = result.sort_values("_poi_row").drop(columns="_poi_row")
result["boundary_year"] = 2021
result["boundary_source"] = "Stockholm Stad / krasch stockholm-stadsdelar (2021)"
result["district_match_status"] = result["district"].notna().map({True:"matched", False:"unmatched"})

assert len(result) == len(pois), "Rows changed unexpectedly after spatial join"
result.to_csv(OUTPUT, index=False, encoding="utf-8-sig")
print(f"Saved {OUTPUT} with {len(result):,} rows")
print("Matched:", result["district"].notna().sum())
print("Unmatched:", result["district"].isna().sum())
print("Points touching multiple polygons:", ambiguous)
print("\nTop districts:")
print(result["district"].value_counts().head(15))
display(result[["name", "analysis_group", "district", "district_match_status"]].head(12))

Saved stockholm_poi_with_districts.csv with 4,234 rows
Matched: 4148
Unmatched: 86
Points touching multiple polygons: 0

Top districts:
district
Södermalm               694
Norrmalm                460
Vasastaden              428
Östermalm               264
Kungsholmen             206
Djurgården              196
Gamla Stan              184
Norra Djurgården         97
Ladugårdsgärdet          85
Södra Hammarbyhamnen     81
Johanneshov              66
Liljeholmen              64
Kista                    59
Årsta                    50
Midsommarkransen         46
Name: count, dtype: Int64


,name,analysis_group,district,district_match_status
0,Stockholms stadshus,culture_heritage,Kungsholmen,matched
1,Stockholms slott,visitor_attraction,Gamla Stan,matched
2,Riksdagshuset,culture_heritage,Gamla Stan,matched
3,Historiska museet,visitor_attraction,Östermalm,matched
4,Arvfurstens palats,culture_heritage,Norrmalm,matched
5,Ålstensskogen,parks_recreation,Ålsten,matched
6,Ulvsunda slott,culture_heritage,Ulvsunda,matched
7,Ekensbergs Båtsällskap,parks_recreation,<NA>,unmatched
8,Bååtska palatset,culture_heritage,Norrmalm,matched
9,Hallwylska palatset,visitor_attraction,Norrmalm,matched


## Current Stockholm stadsdelsområden (11 boroughs, 2025)

The `district` column above represents **smaller 2021 stadsdelar**. This step adds `stadsdelsomrade` for the **11 larger administrative areas** used in your example Airbnb graphic, from an ArcGIS layer labelled `Stadsdelsnämnder_2025`.

The source service is a third-party-hosted ArcGIS feature layer, **not independently authenticated as Stockholm City's official GIS service**. Its actual labels and number of polygons are checked at runtime; the notebook stops if the data does not align with the 11 expected area names. No manual or approximate district assignments are made. Unmatched POIs stay unassigned. This step requires internet access once.


In [5]:
# Add Stockholm's 11 larger stadsdelsområden as a NEW column.
import json
import unicodedata
from urllib.parse import quote

BOROUGH_FILE = Path("stockholm_stadsdelsomraden_2025.geojson")
BOROUGH_OUTPUT = Path("stockholm_poi_with_districts_and_areas.csv")
SERVICE = "https://services2.arcgis.com/uM75dQp2RlOXWzhF/ArcGIS/rest/services/Stadsdelsnämnder_2025/FeatureServer"

expected = {
    "Bromma", "Enskede-Årsta-Vantör", "Farsta", "Hägersten-Älvsjö",
    "Hässelby-Vällingby", "Järva", "Kungsholmen", "Norra innerstaden",
    "Skarpnäck", "Skärholmen", "Södermalm"
}

def normal(s):
    return " ".join(str(s).strip().casefold().replace("–", "-").replace("—", "-").split())

canonical = {normal(s): s for s in expected}

if not BOROUGH_FILE.exists():
    # Discover the layer rather than assuming its ID is zero.
    meta = requests.get(SERVICE, params={"f": "json"}, timeout=40)
    meta.raise_for_status()
    info = meta.json()
    layers = info.get("layers", [])
    if not layers:
        raise RuntimeError("Could not discover polygon layers in the 2025 ArcGIS service")
    found = False
    for layer in layers:
        url = f"{SERVICE}/{layer['id']}/query"
        response = requests.get(url, params={
            "where": "1=1", "outFields": "*", "returnGeometry": "true",
            "outSR": "4326", "f": "geojson"
        }, timeout=90)
        if not response.ok:
            continue
        try:
            payload = response.json()
        except ValueError:
            continue
        if payload.get("type") == "FeatureCollection" and payload.get("features"):
            BOROUGH_FILE.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")
            found = True
            break
    if not found:
        raise RuntimeError("Could not download GeoJSON polygons from the 2025 ArcGIS layer. "
                           "Try opening the service in a browser and export its layer as GeoJSON.")

boroughs_raw = gpd.read_file(BOROUGH_FILE).to_crs("EPSG:4326")
print("2025 boundary fields:", boroughs_raw.columns.tolist())

# Identify the area-name field from the actual data, rather than guessing column names.
name_col = None
for candidate in boroughs_raw.columns:
    if candidate == "geometry":
        continue
    vals = {normal(v) for v in boroughs_raw[candidate].dropna().unique()}
    if vals and vals.issubset(set(canonical)) and len(vals) >= 8:
        name_col = candidate
        break
if name_col is None:
    raise ValueError("Unable to identify the 11 administrative-area names automatically. "
                     "Inspect the printed boundary fields and available values.")

boroughs = boroughs_raw[[name_col, "geometry"]].copy()
boroughs["stadsdelsomrade"] = boroughs[name_col].map(lambda x: canonical.get(normal(x)))
boroughs = boroughs.drop(columns=[name_col]).dropna(subset=["stadsdelsomrade"])
boroughs = boroughs.dissolve(by="stadsdelsomrade", as_index=False)
actual = set(boroughs["stadsdelsomrade"])
if actual != expected:
    raise ValueError(f"Expected all 11 areas; found {sorted(actual)}; missing {sorted(expected-actual)}")

# Rejoin original POIs by coordinates; preserve the 2021 district and all existing columns.
base = result.drop(columns=["stadsdelsomrade", "stadsdelsomrade_match_status"], errors="ignore").reset_index(drop=True)
base["_poi_row"] = base.index
pts = gpd.GeoDataFrame(base, geometry=gpd.points_from_xy(base.longitude, base.latitude), crs="EPSG:4326")
matched = gpd.sjoin(pts, boroughs, how="left", predicate="intersects")
multiples = (matched.groupby("_poi_row")["stadsdelsomrade"].count() > 1).sum()
matched = matched.sort_values(["_poi_row", "stadsdelsomrade"]).drop_duplicates("_poi_row")
result = pd.DataFrame(matched.drop(columns=["geometry", "index_right", "_poi_row"], errors="ignore"))
result["stadsdelsomrade_match_status"] = result["stadsdelsomrade"].notna().map({True:"matched", False:"unmatched"})
result["stadsdelsomrade_boundary_source"] = "ArcGIS Stadsdelsnämnder_2025"
assert len(result) == len(base), "Spatial join unexpectedly changed row count"
result.to_csv(BOROUGH_OUTPUT, index=False, encoding="utf-8-sig")
print(f"Saved {BOROUGH_OUTPUT.resolve()}")
print(f"Matched: {result.stadsdelsomrade.notna().sum():,} / {len(result):,} | overlapping boundary matches: {multiples}")
print(result["stadsdelsomrade"].value_counts(dropna=False).to_string())
display(result[["name", "district", "stadsdelsomrade", "analysis_group"]].head(10))


2025 boundary fields: ['FID', 'Namn', 'Typ', 'Shape__Area', 'Shape__Length', 'geometry']


ValueError: Unable to identify the 11 administrative-area names automatically. Inspect the printed boundary fields and available values.

In [ ]:
# Optional quick overview: POI count per stadsdelsområde and analysis group
counts = (result.groupby(["stadsdelsomrade", "analysis_group"])
          .size().unstack(fill_value=0))
display(counts.assign(total=counts.sum(axis=1)).sort_values("total", ascending=False))


## Interactive HTML map — 11 stadsdelsområden + smaller districts

The large 2025 administrative areas are visible by default. The smaller 2021 districts are an optional map layer. Both appear in POI hover and click details.


In [ ]:
# Interactive map: 2021 Stockholm districts + enriched POIs
# Install if needed: %pip install folium
import folium
from folium.plugins import MarkerCluster
from html import escape
from IPython.display import display, HTML

HTML_OUTPUT = Path("stockholm_poi_district_map.html")

# Reuse 'result' and 'polygons' from the cells above.
map_df = result.copy()
map_df["latitude"] = pd.to_numeric(map_df["latitude"], errors="coerce")
map_df["longitude"] = pd.to_numeric(map_df["longitude"], errors="coerce")
map_df = map_df.dropna(subset=["latitude", "longitude"])

m = folium.Map(location=[59.3293, 18.0686], zoom_start=11, tiles=None, control_scale=True)
# Two optional basemaps. Internet connection and provider access required.
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri", name="Esri streets", overlay=False, control=True
).add_to(m)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap (fallback)", overlay=False).add_to(m)

# The larger 11 boroughs are shown separately from the smaller 2021 districts.
folium.GeoJson(
    boroughs.to_json(), name="11 stadsdelsområden (2025)", show=True,
    style_function=lambda feature: {"color": "#ca4136", "weight": 2.7, "fillColor": "#e78855", "fillOpacity": 0.035},
    highlight_function=lambda feature: {"color": "#af2b2b", "weight": 4, "fillOpacity": 0.12},
    tooltip=folium.GeoJsonTooltip(fields=["stadsdelsomrade"], aliases=["Stadsdelsområde:"])
).add_to(m)

# Show the 2021 smaller districts (optional layer, off initially).
folium.GeoJson(
    polygons.to_json(), name="Smaller stadsdelar (2021)", show=False,
    style_function=lambda feature: {"color": "#2769a5", "weight": 1.6, "fillColor": "#5da5da", "fillOpacity": 0.06},
    highlight_function=lambda feature: {"color": "#e07a2d", "weight": 3, "fillOpacity": 0.15},
    tooltip=folium.GeoJsonTooltip(fields=["district"], aliases=["District (2021):"], sticky=True)
).add_to(m)

colors = {
    "food_drink": "#dc7b39", "nightlife": "#8f4fb8",
    "visitor_attraction": "#1769aa", "culture_heritage": "#b64d5c",
    "parks_recreation": "#42965a", "culture_entertainment": "#b2a03b"
}

def present(value):
    return pd.notna(value) and str(value).strip() != ""

def detail(row, key, label):
    value = row.get(key)
    if not present(value):
        return ""
    return f"<b>{escape(label)}:</b> {escape(str(value))}<br>"

# Separate toggleable marker clusters for each category.
for group in sorted(map_df["analysis_group"].fillna("other").unique()):
    subset = map_df[map_df["analysis_group"].fillna("other") == group]
    layer = folium.FeatureGroup(name=f"{group.replace('_', ' ').title()} ({len(subset):,})", show=True)
    cluster = MarkerCluster().add_to(layer)
    color = colors.get(group, "#666666")
    for _, row in subset.iterrows():
        name = escape(str(row.get("name") or "Unnamed POI"))
        district = escape(str(row.get("district"))) if present(row.get("district")) else "Unmatched"
        kind = escape(str(row.get("primary_subcategory"))) if present(row.get("primary_subcategory")) else escape(str(group))
        area = escape(str(row.get("stadsdelsomrade"))) if present(row.get("stadsdelsomrade")) else "Unmatched"
        tooltip_html = f"<b>{name}</b><br>Type: {kind}<br>Stadsdelsområde: {area}<br>District (2021): {district}"
        popup_html = (f"<div style='min-width:230px;max-width:330px'><b>{name}</b><br>"
                      + f"<b>Stadsdelsområde (2025):</b> {area}<br>"
                      + f"<b>District (2021):</b> {district}<br>"
                      + detail(row, "analysis_group", "Group")
                      + detail(row, "primary_subcategory", "Type")
                      + detail(row, "best_website", "Website")
                      + detail(row, "opening_hours", "Opening hours")
                      + detail(row, "wheelchair", "Wheelchair")
                      + detail(row, "all_pois_within_500m", "Other POIs within 500 m")
                      + detail(row, "nearest_visitor_attraction_m", "Nearest attraction (m)")
                      + detail(row, "osm_url", "OSM URL")
                      + "</div>")
        folium.CircleMarker(
            location=[float(row["latitude"]), float(row["longitude"])],
            radius=5, color=color, fill=True, fill_color=color, fill_opacity=0.8,
            weight=1, tooltip=folium.Tooltip(tooltip_html, sticky=True),
            popup=folium.Popup(popup_html, max_width=365)
        ).add_to(cluster)
    layer.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m.save(str(HTML_OUTPUT))
print(f"Saved {HTML_OUTPUT.resolve()} | {len(map_df):,} POIs plotted")
print("Hover for name, type, and both district levels; click for more details; toggle layers at top-right.")
display(m)  # If Jupyter blocks this, open the saved HTML in your browser.
